Notebook to inspect .dm3-files

## Contents

 1. [Load](#1.-Load)
 2. [Inspect](#2.-Inspect)
 3. [Parameters](#3.-Parameters)
 4. [Calibrate](#4.-Calibrate)
 5. [Scalebar](#5.-Scalebar)
 6. [Plot](#6.-Plot)

In [6]:
%matplotlib tk
import hyperspy.api as hs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patheffects as path_effects
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from scipy.ndimage import rotate as rotate
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

#Import path handling tool
from pathlib import Path

# 1. Load

In [44]:
base_directory = Path(f"/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4")
files = list(base_directory.glob("*.dm3"))
print("Found files:")
for i, f in enumerate(files):
    print(f"{i}:/t{f}")

Found files:
0:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/diff.dm3
1:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/nw-bf.dm3
2:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/df-aperture-underfocus.dm3
3:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/hrtem-mag-1,2.dm3
4:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/diff-nw-single-crystal.dm3
5:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/df-aperture-focus.dm3
6:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/nw-df-objective-aperture.dm3
7:/t/Users/ingridmzg/Library/CloudStorage/OneDrive-NTNU/Våren 2026/Bilder/JEOL2100/2026_03_13_Master_B4/diff-

In [45]:
# Select indices from the printed list
s = files[22]
dp = files[7]

In [46]:
#LOAD
s = hs.load(str(s))
dp = hs.load(str(dp))

# 2. Inspect

press keyboard key "h" to adjust brightness

In [ ]:
s.plot(axes_off=True)

invalid command name "6292313984idle_draw"
    while executing
"6292313984idle_draw"
    ("after" script)


### Contrast Settings

In [ ]:
# set contrast values after inspection above
contrast_settings = {
    "s" : {"vmin": 5000, "vmax": 12000},
    "dp" : {"vmin": -100, "vmax": 800},
}

### Rotate

In [ ]:
#rotate to see
angle = 0
rotated_s = s.map(rotate, angle=angle, reshape=False, inplace=False)

In [ ]:
rotated_s.plot()

# 3. Parameters

In [35]:
s.set_signal_type("electron_diffraction")

/Users/ingridmzg/Code/Master_Code/.venv/lib/python3.10/site-packages/hyperspy/misc/utils.py:72: VisibleDeprecationWarning: `_get_block_pattern` has moved to `hyperspy.misc.dask_utils`. It is for internal use only and may be removed in the future.
  warnings.warn(


In [36]:
s.data.dtype

dtype('float32')

In [37]:
s.metadata

├── Acquisition_instrument
│   └── TEM
│       ├── acquisition_mode = TEM
│       ├── beam_current = 0.0
│       ├── beam_energy = 200.0
│       ├── magnification = 50000.0
│       └── microscope = JEOL COM
├── General
│   ├── FileIO
│   │   └── 0
│   │       ├── hyperspy_version = 2.4.0
│   │       ├── io_plugin = rsciio.digitalmicrograph
│   │       ├── operation = load
│   │       └── timestamp = 2026-03-16T13:09:35.461994+01:00
│   ├── date = 2026-03-13
│   ├── original_filename = 3_df_nw.dm3
│   ├── time = 11:44:20
│   └── title = 3_df_nw
└── Signal
    ├── Noise_properties
    │   └── Variance_linear_model
    │       ├── gain_factor = 1.0
    │       └── gain_offset = 0.0
    ├── quantity = Intensity
    └── signal_type = electron_diffraction

# 4. Calibrate

calibrate the images based on dp

In [51]:
dp.plot(cmap="magma", norm="symlog", vmin="70")

line = hs.roi.Line2DROI(x1=2, y1=1, x2=300, y2=300, linewidth=1)
profile = line.interactive(dp)
profile.plot(norm="log")

span = hs.roi.SpanROI()
span.add_widget(profile)

In [56]:
a_GaAs = 5.653 #Å
h, k, l = 1, 1, 1
n = 8
hkl = np.array([h, k , l])*n
g_hkl = np.sqrt(np.sum((hkl/a_GaAs)**2))
L = (span.right-span.left) / profile.axes_manager[0].scale
scale = g_hkl / L
print(f" Lattice spacing of ({h*n}, {k*n}, {l*n}) is {g_hkl:.3g} 1/Å \n Measured distance on the detector is {L:.2f}\n Diffraction scale is {scale:.3g} 1/Å")

 Lattice spacing of (8, 8, 8) is 2.45 1/Å 
 Measured distance on the detector is 1353.93
 Diffraction scale is 0.00181 1/Å


In [ ]:
dp.axes_manager[0].scale = scale
dp.axes_manager[1].scale = scale
dp.axes_manager[0].units = "Å⁻¹"
dp.axes_manager[1].units = "Å⁻¹"

# 5. Scalebar

In [ ]:
def get_pixel_size(signal):

    axis = signal.axes_manager[0]
    return float(axis.scale) #nm/pixel

In [ ]:
def generate_scalebar(ax, size, signal, label_units="nm", pos = "lower left"):
    fontprops = fm.FontProperties(size=8)
    scale = signal.axes_manager[0].scale
    size_in_pixels = size / scale

    scalebar = AnchoredSizeBar(
        transform = ax.transData,
        size = size_in_pixels,
        label = f"{size:.g} {label_units}",
        loc = pos,
        frameon = False,
        color = "white",
        size_vertical = 2,
        pad = 0.3,
        fontproperties = fontprops
    )

    ax.add_artist(scalebar)
    return scalebar

# 6. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
s = hs.signals.Signal2D(rotate(s.data, angle=0, reshape=False))
ax.imshow(s.data, cmap="magma", origin="lower", vmin=contrast_settings["s"]["vmin"], vmax=contrast_settings["s"]["vmax"])
generate_scalebar(ax, size=5, signal=s, label_units="1/nm"), pos = "lower left"
ax.axis("off")
plt.show()